In [83]:
from google.cloud import storage
import os
from dotenv import load_dotenv
import pandas as pd
from io import BytesIO

load_dotenv()

True

In [84]:
GCS_BUCKET = os.getenv('GCS_BUCKET').strip()
GCS_MEDAL_FETCH = os.getenv('GCS_MEDAL_FETCH').strip()
SEASON = int(os.getenv('SEASON'))
BLOB_FETCH = f"{GCS_MEDAL_FETCH}/season={SEASON}/laps/all_session_laps.parquet"

REFERENCE_BLOB_BASE = f"gold/season={SEASON}"
BLOB_FETCH_SESSIONS = f"{REFERENCE_BLOB_BASE}/session.parquet"
BLOB_FETCH_DRIVERS = f"{REFERENCE_BLOB_BASE}/driver.parquet"

In [85]:
def load_client_bucket(bucket):
    if not bucket:
        raise ValueError("[ERROR] Bucket Name is Required")
    client = storage.Client()
    bucket = client.bucket(bucket)
    return client, bucket

def load_parquet(blob):
    print(f"[INFO] Loading Parquet from GCS: {blob.name}")
    df = pd.read_parquet(BytesIO(blob.download_as_bytes()))
    print(f"[INFO] Loaded Parquet from GCS: {blob.name}")
    return df

def load_parquet_from_gcs(bucket, blob_path):
    if not blob_path:
        raise ValueError("[ERROR] Blob Path is Required")
    blob = bucket.blob(blob_path)
    if not blob.exists():
        raise FileNotFoundError(f"[ERROR] Blob does not exist: {blob_path}")
    df = load_parquet(blob)
    return df

def drop_columns(df, columns):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        print(f"[WARN] Columns not found in df: {missing}")

    print(f"[INFO] Dropping columns: {columns}")
    df = df.drop(columns=columns, errors="ignore")
    print(f"[INFO] Columns Dropped")
    return df

def strip_string(column):
    if column is None:
        raise ValueError("[ERROR] Column not found")
    
    column = column.str.strip()
    print(f"[INFO] Stripping String")
    return column

In [86]:
client, bucket = load_client_bucket(GCS_BUCKET)

In [87]:
df = load_parquet_from_gcs(bucket, BLOB_FETCH)
df_sessions = load_parquet_from_gcs(bucket, BLOB_FETCH_SESSIONS)
df_drivers = load_parquet_from_gcs(bucket, BLOB_FETCH_DRIVERS)

[INFO] Loading Parquet from GCS: bronze/season=2026/laps/all_session_laps.parquet
[INFO] Loaded Parquet from GCS: bronze/season=2026/laps/all_session_laps.parquet
[INFO] Loading Parquet from GCS: gold/season=2026/session.parquet
[INFO] Loaded Parquet from GCS: gold/season=2026/session.parquet
[INFO] Loading Parquet from GCS: gold/season=2026/driver.parquet
[INFO] Loaded Parquet from GCS: gold/season=2026/driver.parquet


In [88]:
cols_to_drop1 = [
        "date_start",
        "meeting_key",
        "circuit_key",
        "circuit_short_name",
        "location",
        "gmt_offset",
        "year",
        "ingested_at",
        "country_key",
        "country_code",
        "country_name",
]

columns_to_drop2 = [
        "broadcast_name",
        "name_acronym",
        "team_colour",
        "first_name",
        "last_name",
        "headshot_url",
]

df_sessions_right = drop_columns(df_sessions, cols_to_drop1)
df_drivers_right = drop_columns(df_drivers, columns_to_drop2)

[INFO] Dropping columns: ['date_start', 'meeting_key', 'circuit_key', 'circuit_short_name', 'location', 'gmt_offset', 'year', 'ingested_at', 'country_key', 'country_code', 'country_name']
[INFO] Columns Dropped
[INFO] Dropping columns: ['broadcast_name', 'name_acronym', 'team_colour', 'first_name', 'last_name', 'headshot_url']
[INFO] Columns Dropped


In [89]:
print(f"[INFO] Merging laps and sessions on session_key")
new_df = pd.merge(df, df_sessions_right, on="session_key", how="left")

print(f"[INFO] Merging laps and drivers on driver_number")
new_df = pd.merge(new_df, df_drivers_right, on="driver_number", how="left")

print(f"[INFO] Merging success")

column_order = [
    "meeting_key", 
    "session_key", 
    "session_name",
    "driver_number", 
    "full_name",
    "team_name",
    "country",
    "lap_number", 
    "date_start", 
    "duration_sector_1", 
    "duration_sector_2",
    "duration_sector_3",
    "i1_speed",
    "i2_speed",
    "lap_duration",
    "segments_sector_1",
    "segments_sector_2",
    "segments_sector_3",
    "st_speed",
    "ingested_at"
]

new_df = new_df[column_order]

[INFO] Merging laps and sessions on session_key
[INFO] Merging laps and drivers on driver_number
[INFO] Merging success


In [90]:
string_strip_cols = ["session_name", "full_name", "team_name", "country"]
for c in string_strip_cols:
    print(f"[INFO] Stripping {c}")
    new_df[c] = strip_string(new_df[c])

[INFO] Stripping session_name
[INFO] Stripping String
[INFO] Stripping full_name
[INFO] Stripping String
[INFO] Stripping team_name
[INFO] Stripping String
[INFO] Stripping country
[INFO] Stripping String


In [91]:
session_date_map = (
    df_sessions.drop_duplicates(subset=["session_key"]).set_index("session_key")["date_start"]
)

In [92]:
new_df["date_start"] = new_df["date_start"].fillna(
    new_df["session_key"].map(session_date_map)
)

In [93]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18031 entries, 0 to 18030
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   meeting_key        18031 non-null  int64              
 1   session_key        18031 non-null  int64              
 2   session_name       18031 non-null  object             
 3   driver_number      18031 non-null  int64              
 4   full_name          18020 non-null  object             
 5   team_name          18020 non-null  object             
 6   country            18020 non-null  object             
 7   lap_number         18031 non-null  int64              
 8   date_start         18031 non-null  object             
 9   duration_sector_1  16185 non-null  float64            
 10  duration_sector_2  17807 non-null  float64            
 11  duration_sector_3  17472 non-null  float64            
 12  i1_speed           16070 non-null  float64    

In [94]:
new_df.head()

,meeting_key,session_key,session_name,driver_number,full_name,team_name,country,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,lap_duration,segments_sector_1,segments_sector_2,segments_sector_3,st_speed,ingested_at
0,1304,11465,Testing Day 1,31,Esteban OCON,Haas F1 Team,France,1,2026-02-11 07:00:00+00:00,NaN,NaN,24.319,NaN,NaN,99.848,None,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",NaN,2026-05-04 18:23:14.237848+00:00
1,1304,11465,Testing Day 1,5,Gabriel BORTOLETO,Audi,Brazil,1,2026-02-11 07:00:00+00:00,32.258,44.634,27.954,220.0,237.0,104.846,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[2049.0, 2049.0, 2048.0, 2048.0, 2049.0, 2049....","[2048.0, 2048.0, 2048.0, 2048.0, 2048.0, 2048....",NaN,2026-05-04 18:23:14.237848+00:00
2,1304,11465,Testing Day 1,41,Arvid LINDBLAD,Racing Bulls,United Kingdom,1,2026-02-11 07:00:00+00:00,32.406,45.046,24.557,224.0,224.0,102.009,"[nan, nan, nan, nan, nan, nan, nan, 2048.0, 20...","[2048.0, 2049.0, 2048.0, 2048.0, 2048.0, 2048....","[2048.0, 2048.0, 2048.0, 2048.0, 2048.0, 2048....",NaN,2026-05-04 18:23:14.237848+00:00
3,1304,11465,Testing Day 1,55,Carlos SAINZ,Williams,Spain,1,2026-02-11 07:00:00+00:00,NaN,54.802,28.621,178.0,182.0,392.014,"[nan, 2064.0, 2064.0, 2048.0, 2048.0, 2048.0, ...","[2048.0, 2048.0, 2048.0, 2048.0, 2048.0, 2048....","[2048.0, 2048.0, 2048.0, 2048.0, 2048.0, 2048....",178.0,2026-05-04 18:23:14.237848+00:00
4,1304,11465,Testing Day 1,81,Oscar PIASTRI,McLaren,Australia,1,2026-02-11 07:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,None,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",NaN,2026-05-04 18:23:14.237848+00:00
